In [4]:
import os, sys, math, ssl, io, pytz, numpy as np, pandas as pd, requests
from datetime import datetime, timedelta, date
from timezonefinder import TimezoneFinder
from meteostat import Stations, Hourly
from isd import Batch
from scp import SCPClient
import paramiko
import calendar
from pandas.errors import EmptyDataError

from methods import *



# Define constants
year = 2024
file_type = 'AMY'
save_folder = f'epws_wmo_{year}'

# Check if the 'zipcodes' variable is already defined
if 'zipcodes' not in globals():
    # Load the zip codes CSV only if 'zipcodes' is not already defined
    zipcodes = pd.read_csv(f'resources/zip_code_list_{year}.csv', dtype={f'EPW_file_name_{year}': str, f'weather_station_wmo_{year}': str})

# Initialize a counter for iterations
counter = 0

# Process each row in the DataFrame starting from the specified index
for index, row in zipcodes.iloc[16550:].iterrows():
    # if float(row.get(f"distance_location_station_miles_{year}")) < 50:
    #     continue
    print(index)


    zip_code = str(row['zip0']).zfill(5)  # Ensure the zip code is a string and pad with leading zeros if needed
    print(zip_code)
    lat = row['lat']
    lon = row['lng']
    save_name = None

    # print(lat)
    # print(lon)
    # print(row['city'])

    # Retrieve data for the current location
    # retrieve_status, distance, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations = run_individual_location(lat, lon, year, file_type, save_folder, save_name)
    retrieve_status, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations, flags = run_individual_location(lat, lon, year, file_type, save_folder, save_name)
    if retrieve_info_closest_other_locations:
        try:
            # retrieve_status, distance, hdd, cdd = retrieve_info_other_location(wmo, zipcodes, year)
            retrieve_status, hdd, cdd, flags= retrieve_info_other_location(wmo, zipcodes, year)
            # print(retrieve_status)
            # print('=====')
            # print(wmo)
        except IndexError:
            # retrieve_status, distance, hdd, cdd = retrieve_info_other_location(wmo, zipcodes, year)
            print('**********')
            print(wmo)
            retrieve_status, hdd, cdd, flags = retrieve_info_other_location(wmo, zipcodes, year)





    # Validate that retrieve_status is a boolean
    if not isinstance(bool(retrieve_status), bool):
        raise TypeError(f"retrieve_status is not a boolean. Actual value: {retrieve_status}. Program stopped.")
    
    [distance_mi, lat_station, lon_station] = retrieve_distance_station_location(wmo, lat, lon)

    # Update the DataFrame only if the cell is empty or contains a placeholder (like 'nan')
    update_if_missing(zipcodes, index, f"EPW_file_name_{year}", f"{wmo}_{year}.epw")
    update_if_missing(zipcodes, index, f"distance_location_station_miles_{year}", distance_mi)  # Convert from meters to miles
    update_if_missing(zipcodes, index, f"weather_station_wmo_{year}", wmo)
    update_if_missing(zipcodes, index, f"hdd_base65F_{year}", hdd)
    update_if_missing(zipcodes, index, f"cdd_base65F_{year}", cdd)
    update_if_missing(zipcodes, index, f"Tdb_holes_{year}", flags[6])
    update_if_missing(zipcodes, index, f"Tdew_holes_{year}", flags[7])
    update_if_missing(zipcodes, index, f"RH_holes_{year}", flags[8])

    # Increment the counter
    counter += 1

    # Every 10 iterations, save the DataFrame and reopen it
    if counter % 10 == 0:
        # Save the DataFrame to the CSV file
        zipcodes.to_csv(f'resources/zip_code_list_{year}.csv', index=False)

        # Reopen the file to ensure the latest version is loaded
        # zipcodes = pd.read_csv('resources/zip_code_list.csv', dtype={f'Do we have data for {year}?': str, f'weather_station_wmo_{year}': str})

# After the loop is done, ensure the latest state is saved
zipcodes.to_csv(f'resources/zip_code_list_{year}.csv', index=False)

# KVHN0
# KSUN
# 74611


16550
33867
16551
53150
16552
62330
16553
15628
16554
35173
16555
28460
16556
97267
16557
92341
16558
79527
16559
79366
16560
53937
16561
46958
16562
42366
16563
02038
16564
42369
16565
62890
16566
43143
16567
44507
16568
47863
16569
61282
16570
55909
16571
08854
16572
70715
16573
54173
16574
40444
16575
61928
16576
17016
16577
40110
16578
37184
16579
41647
16580
06234
16581
60452
16582
01354
16583
77880
16584
79562
16585
65053
16586
68723
16587
18371
16588
77518
16589
01863


16590
05764
16591
02025


16592
45769
16593
55792
16594
78142
16595
56467
16596
38134
16597
02631


16598
58481
16599
04353
16600
38006
16601
45337
16602
29105
16603
48130
16604
47041
16605
50263
16606
91360
16607
38229
16608
04021
16609
62666
16610
78736
16611
01054
16612
94115


16613
61552
16614
34983
16615
48861
16616
03102
16617
14526
16618
22903
16619
83615
16620
25414
16621
26378
16622
47336
16623
73764
16624
62255
16625
56166
16626
62548
16627
04280
16628
65754
16629
71842
16630
01569
16631
67212
16632
16635
16633
72935
16634
28463
16635
19562
16636
28031
16637
56137
16638
98177
16639
95409
16640
47163
16641
80123
16642
15324
16643
95689
16644
60202
16645
16110
16646
63841
16647
15957
16648
36572
16649
97826
16650
63126
16651
99603
16652
38483
16653
10953
16654
49801
16655
25904
16656
54623


...

29757
45342
29758
73127
29759
14305
29760
47129
29761
23847
29762
53039
29763
11557
29764
22841
29765
19023
29766
29375
29767
16673
29768
08869
29769
16037
29770
05151
29771
66762
29772
45227
29773
29582
29774
16732
29775
11714
29776
72082
29777
75182
29778
37777
29779
40204
29780
85395
29781
32789
29782
20746
29783
14215
29784
89030
29785
49686
29786
59330
29787
01983
29788
23885
29789
76114
29790
76309
29791
39206
29792
79904
29793
50025
29794
40205
29795
13039
29796
64136
29797
33066
29798
12177
29799
58421
29800
84405
29801
48183
29802
64093
29803
14551
29804
73401
29805
04962
29806
82513
29807
93463
29808
43733
29809
45206
29810
61068
29811
95835
29812
32127
29813
33308
29814
29172
29815
66078
29816
45207
29817
98144
29818
65739
29819
46738
29820
06105
29821
02743
29822
02026
29823
01056
29824
76502
29825
91915
29826
25287
29827
71104
29828
72601
29829
19608
29830
15676
29831
51639
29832
33913
29833
48098
29834
23093
29835
91706
29836
53559
29837
30168
29838
50117
29839
66223
2984

30411
93962
30412
77581
30413
21211
30414
67735
30415
11213
30416
98407
30417
40214
30418
64163
30419
62521
30420
48446
30421
95111
30422
91710
30423
96707
30424
92315
30425
58402
30426
25550
30427
78719
30428
41659
30429
78577
30430
95824
30431
97212
30432
78645
30433
13310
30434
68178
30435
37916
30436
37917
30437
76574
30438
31501
30439
98466
30440
11226
30441
07064
30442
76548
30443
56482
30444
27559
30445
47305
30446
23451
30447
29033
30448
46556
30449
33629
30450
43214
30451
43050
30452
68850
30453
94041
30454
29169
30455
47542
30456
60115
30457
03766
30458
99504
30459
14742
30460
24311
30461
32724
30462
37210
30463
28349
30464
19604
30465
80919
30466
31906
30467
07087
30468
97301
30469
48846
30470
89005
30471
28147
30472
92606
30473
29301
30474
92081
30475
56560
30476
02118
30477
46938
30478
48642
30479
30054
30480
19464
30481
37854
30482
77504
30483
20006
30484
49022
30485
99006
30486
91602
30487
76543
30488
92106
30489
81615
30490
18017
30491
39309
30492
45424
30493
43613
3049

32249
78596
32250
71923
32251
30144
32252
54929
32253
23401
32254
89108
32255
92011
32256
67642
32257
64106
32258
90045
32259
35950
32260
33760
32261
53144
32262
97211
32263
78041
32264
10172
32265
60070
32266
24957
32267
92880
32268
99546


32269
05819
32270
75233
32271
79789
32272
19609
32273
92553
32274
18234
32275
08251
32276
56172
32277
83460
32278
94065
32279
21672
32280
74354
32281
78756
32282
55371
32283
98444
32284
34102
32285
33907
32286
02745
32287
27834
32288
34243
32289
98501
32290
95831
32291
90807
32292
99655
32293
29904
32294
10171
32295
17034
32296
88330
32297
98499
32298
18970
32299
20740
32300
77571
32301
21220
32302
70760
32303
91780
32304
07006
32305
07961
32306
10029
32307
11215
32308
07981
32309
02888
32310
93010
32311
42533
32312
90704
32313
13602
32314
52242
32315
96860
32316
52632
32317
98166
32318
48413
32319
65789
32320
37217
32321
26505
32322
97002
32323
94043
32324
78703
32325
38116
32326
91606
32327
20755
32328
29115
32329
04970
32330
37701
32331
79111
32332
90245
32333
46321
32334
94535
32335
31315
32336
90011
32337
53501
32338
10044
32339
10002
32340
95625
32341
99722
32342
32225
32343
80239
32344
37915
32345
76048
32346
99626
32347
62460
32348
32925
32349
30906
32350
32780
32351
95110
3235

32577
17070
32578
48084
32579
96818
32580
32544
32581
62821
32582
27863
32583
73132
32584
99658
32585
83121
32586
21230
32587
54545
32588
10162
32589
81506
32590
73624
32591
78243
32592
30401
32593
51510
32594
58705
32595
23824
32596
29605
32597
54455
32598
33607
32599
58072
32600
96857
32601
99659
32602
55358
32603
86015
32604
11975
32605
47432
32606
53207
32607
57730
32608
07643
32609
43465
32610
85286
32611
20737
32612
02062
32613
89423
32614
29108
32615
62881
32616
31634
32617
99506
32618
75402
32619
80238
32620
78657
32621
64153
32622
92833
32623
33062
32624
76448
32625
33859
32626
75603
32627
64601
32628
96093
32629
99569
32630
47324
32631
62056
32632
75087
32633
33126
32634
23337
32635
27201
32636
93524
32637
10069
32638
33462
32639
10282
32640
99614
32641
58102
32642
32803
32643
25530
32644
74464
32645
50321
32646
32548
32647
23604
32648
11096
32649
33050
32650
49782
32651
35476
32652
75237
32653
55008
32654
11217
32655
72022
32656
23185
32657
22202
32658
55313
32659
99558
3266

In [1]:
import os, sys, math, ssl, io, pytz, numpy as np, pandas as pd, requests
from datetime import datetime, timedelta, date
from timezonefinder import TimezoneFinder
from meteostat import Stations, Hourly
from isd import Batch
from scp import SCPClient
import paramiko
import calendar
from pandas.errors import EmptyDataError

from methods import *

# Define constants
year = 2023
file_type = 'AMY'
save_folder = f'epws_wmo_{year}'

# Check if the 'zipcodes' variable is already defined
if 'zipcodes' not in globals():
    # Load the zip codes CSV only if 'zipcodes' is not already defined
    zipcodes = pd.read_csv(f'resources/zip_code_list_{year}.csv', 
                           dtype={f'EPW_file_name_{year}': str, 
                                  f'weather_station_wmo_{year}': str})

# Initialize a counter for iterations
counter = 0

# Process each row in the DataFrame where 'EnergyPlus Status' is 'Bad'
for index, row in zipcodes[zipcodes['EnergyPlus Status'] == 'Bad'].iterrows():
    
    print(index)

    zip_code = str(row['zip0']).zfill(5)  # Ensure the zip code is a string and pad with leading zeros if needed
    print(zip_code)
    lat = row['lat']
    lon = row['lng']
    save_name = None

    # Retrieve data for the current location
    retrieve_status, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations, flags = \
        run_individual_location(lat, lon, year, file_type, save_folder, save_name)
    
    if retrieve_info_closest_other_locations:
        try:
            retrieve_status, hdd, cdd, flags = retrieve_info_other_location(wmo, zipcodes, year)
        except IndexError:
            retrieve_status, hdd, cdd, flags = retrieve_info_other_location(wmo, zipcodes, year)

    # Validate that retrieve_status is a boolean
    if not isinstance(bool(retrieve_status), bool):
        raise TypeError(f"retrieve_status is not a boolean. Actual value: {retrieve_status}. Program stopped.")
    
    # Calculate distance between station and location
    distance_mi, lat_station, lon_station = retrieve_distance_station_location(wmo, lat, lon)

    # Update the DataFrame only if the cell is empty or NaN
    update_if_missing(zipcodes, index, f"EPW_file_name_{year}", retrieve_status)
    update_if_missing(zipcodes, index, f"distance_location_station_miles_{year}", distance_mi)
    update_if_missing(zipcodes, index, f"weather_station_wmo_{year}", wmo)
    update_if_missing(zipcodes, index, f"hdd_base65F_{year}", hdd)
    update_if_missing(zipcodes, index, f"cdd_base65F_{year}", cdd)
    update_if_missing(zipcodes, index, f"Tdb_holes_{year}", flags[6])
    update_if_missing(zipcodes, index, f"Tdew_holes_{year}", flags[7])
    update_if_missing(zipcodes, index, f"RH_holes_{year}", flags[8])

    # Increment the counter
    counter += 1

    # Every 10 iterations, save the DataFrame and reopen it
    if counter % 10 == 0:
        zipcodes.to_csv(f'resources/zip_code_list_{year}.csv', index=False)
        # Reopen the file if needed
        # zipcodes = pd.read_csv(f'resources/zip_code_list_{year}.csv', 
        #                       dtype={f'EPW_file_name_{year}': str, 
        #                              f'weather_station_wmo_{year}': str})

# After the loop is done, ensure the latest state is saved
zipcodes.to_csv(f'resources/zip_code_list_{year}.csv', index=False)


Debug

In [3]:

def get_noaa_merra2_data(lat, lon, year, file_type, save_folder):
    """
    Retrieves NOAA and MERRA2 data for a specific location and year.
    """
    retrieve_status = True
    data_noaa, tz, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists, incomplete_timeseries = get_data_noaa(lat, lon, year, save_folder)
    # data_noaa, tz, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists, incomplete_timeseries = get_data_noaa(lat, lon, year, save_folder)
    if epw_exists:
        df_merged = ''
        retrieve_status = False
        # distance = ''
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = ''
        epw_exists = True
        # return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
        return df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
    elif incomplete_timeseries:
        df_merged = ''
        retrieve_status = False
        # distance = np.nan
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = '' 
        wmo = ''
        epw_exists = False
        # return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
        return df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists

    try:
        data_noaa_tz_adj = filter_dataframe_by_date(convert_utc_to_local(data_noaa, tz), datetime(year, 1, 1), datetime(year+1, 1, 1))
    except AttributeError:
        # print("We don't have NOAA data for this location/year")
        df_merged = ''
        retrieve_status = False
        # distance = np.nan
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = '' 
        epw_exists = False
        # return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
        return df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists

    info_dict = {
    'timeshift': get_time_shift(tz),
    'elevation': elevation,
    'wmo': wmo,
    'station_name': station_name,
    'state': state,
    'country': country,
    'lat': latitude_station,
    'lon': longitude_station,
    'weather_file_type': file_type
    }

    data_noaa_tz_adj_h = data_noaa_tz_adj.resample('H').mean()
    data_noaa_tz_adj_h_interpolated = data_noaa_tz_adj_h.interpolate(method='linear', limit=3, limit_direction='both')
    hdd, cdd = calculate_hdd_cdd(data_noaa_tz_adj_h_interpolated, 'temp')
    try:
        df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)
    except EmptyDataError:
        try:
            df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)
        except EmptyDataError:
            try:
                df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)
            except EmptyDataError:
                try:
                    df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)
                except EmptyDataError:
                    df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)


    df_merged = merge_data(df_merra2, data_noaa_tz_adj_h_interpolated)

    # Check for empty cells in df_merged
    # if df_merged.isnull().any().any():
    #     raise ValueError("The merged DataFrame (df_merged) contains empty cells. Stopping execution.")


    # return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
    return df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists


[df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists] = get_noaa_merra2_data(37.73096, -115.24786, 2022, 'AMY', '')





In [5]:
df_merged.to_csv('TEST_.csv')

In [ ]:
print(zipcodes.columns)

In [ ]:
meteostat_df = pd.read_csv('resources/meteostat_stats.csv')
zipcodes = pd.read_csv('resources/zip_code_list.csv')
meteostat_df


In [ ]:
zip_row.columns

In [ ]:
import math

# Function to calculate the distance between two lat/lon points using the Haversine formula
def haversine(lat1, lon1, lat2, lon2):
    # Radius of the Earth in miles
    R = 3958.8
    
    # Convert degrees to radians
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)
    
    # Haversine formula
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    # Distance in miles
    return R * c


# Iterate over each row in zip_df
for idx, zip_row in zipcodes.iterrows():
    # print(idx)

    if bool(zip_row['Do we have data for 2022?']):
        wmo_code = zip_row['weather_station_wmo_2022']

        # print(wmo_code)
        
        # Look for a matching row in meteostat_df using WMO or ICAO code
        # station_row = meteostat_df[(meteostat_df['wmo'].astype(str) == wmo_code) | (meteostat_df['icao'].astype(str) == wmo_code)]
        try:
            station_row = meteostat_df[(meteostat_df['id'].astype(str) == str(wmo_code))]
        except ValueError:
            try:
                station_row = meteostat_df[(meteostat_df['id'].astype(str) == wmo_code[:4])]
            except TypeError:
                print('----------------------------')
                print(wmo_code)
                print(zip_row['zip0'])
                print(zip_row['Do we have data for 2022?'])
                continue
                # station_row = meteostat_df[(meteostat_df['icao'].astype(str) == wmo_code[:4])]
               
        



        if not station_row.empty:
            # Extract lat/lon from meteostat_df
            lat_stat = station_row['latitude'].values[0]
            lon_stat = station_row['longitude'].values[0]
            
            # Extract lat/lon from zip_df
            lat_zip = zip_row['lat']
            lon_zip = zip_row['lng']
            
            # Calculate the distance in miles
            distance = haversine(lat_zip, lon_zip, lat_stat, lon_stat)
            
            # Save the calculated distance in the new column
            zipcodes.at[idx, 'distance_location_station_miles_2022___'] = distance

# Show the updated zip_df with distances
zipcodes.head()


In [ ]:
meteostat_df[(meteostat_df['wmo'] == int('71345'))]

In [7]:
zipcodes.to_csv('resources/zip_code_list.csv', index=False)


In [19]:


Stations().nearby(48.72526, -111.36528).fetch().to_csv('resources/meteostat_stats.csv', index=True)

In [ ]:
def get_data_noaa(lat, lon, year, save_folder):
    """
    Fetches NOAA data for a given location and year, handling timezones and missing data.
    """
    # Disable SSL verification
    ssl._create_default_https_context = ssl._create_unverified_context

    start = datetime(year - 1, 12, 31)
    end = datetime(year + 1, 1, 2)

    stations = Stations().nearby(lat, lon)

    epw_exists = False
    station_number = 0
    len_data = 0

    incomplete_timeseries = True
    while incomplete_timeseries:
        station_number += 1
        wmo = fix_wmo(str(stations.fetch(station_number).index.values[-1]))
        # First check if EPW already exists
        if check_epw_exists(save_folder, year, wmo):
            epw_exists = True
            incomplete_timeseries = False
            break
        data = Hourly(stations.fetch(station_number), start, end, model=True).fetch()
        len_data = len(data)
        missing_hours_num, largest_consecutive_group = check_missing_hours(year, data)
        if (len_data > 8000) & (largest_consecutive_group <= 3):
            incomplete_timeseries = False
        # distance = stations.fetch(station_number)['distance'].values[-1]
        # # Let's stop after 100mi
        # if distance > 160000:
        #     break

    if epw_exists | incomplete_timeseries:
        data = ''
        timezone = ''
        # distance = ''
        elevation = ''
        station_name = ''
        state = ''
        country = ''
        latitude_station = ''
        longitude_station = ''
                
    else:
        station_info = stations.fetch(station_number)
        timezone = station_info['timezone'].values[-1]
        elevation = station_info['elevation'].values[-1]
        # distance = stations.fetch()['distance'].values[-1]
        wmo = fix_wmo(str(station_info.index.values[-1]))
        station_name = station_info['name'].values[-1]
        state = station_info['region'].values[-1]
        country = station_info['country'].values[-1]
        latitude_station = station_info['latitude'].values[-1]
        longitude_station = station_info['longitude'].values[-1]

    # return data, timezone, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries
    return data, timezone, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries



lat = 42.06259
lon = -72.62589

data, timezone, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries = get_data_noaa(lat, lon, 2022, '')
data.head(5)

In [ ]:
data.interpolate(method='linear', limit=3, limit_direction='forward')

In [ ]:
zipcodes.head(20)

In [ ]:

file_type = 'AMY'
lat = 41.766595
lon = -88.318735
year = 2023
name= 'TEST_2023'
output_name = name + '.epw'

# Run your existing code with these parameters
data_meteostat_merra2, missing_dates, info_dict = get_noaa_merra2_data(lat, lon, year, file_type)
data_meteostat_merra2.to_csv(output_name, header=False, index=False)
with open(output_name, 'r') as original_file:
    data_content = original_file.read()
header_lines = create_header(data_meteostat_merra2, year, info_dict)
with open(output_name, 'w') as new_file:
    new_file.write("\n".join(header_lines) + "\n" + data_content)


In [ ]:
import sys
from PyQt5.QtWidgets import QApplication, QWidget, QLabel, QLineEdit, QPushButton, QVBoxLayout, QGridLayout, QMessageBox
import pandas as pd
import numpy as np
import paramiko
from scp import SCPClient
from isd import Batch
from meteostat import Stations, Hourly
from timezonefinder import TimezoneFinder
from datetime import datetime, timedelta, date
import pytz
import requests
import ssl
import io

# Assuming all the previous functions are defined above or imported from another module


def run_individual_location(output_name, lat, lon, year, file_type, name):
    try:
        # Run your existing code with these parameters
        data_meteostat_merra2, missing_dates, info_dict = get_noaa_merra2_data(lat, lon, year, file_type)
        data_meteostat_merra2.to_csv(output_name, header=False, index=False)
        with open(output_name, 'r') as original_file:
            data_content = original_file.read()
        header_lines = create_header(data_meteostat_merra2, year, info_dict)
        with open(output_name, 'w') as new_file:
            new_file.write("\n".join(header_lines) + "\n" + data_content)
        QMessageBox.information(window, "Success", f"Data saved successfully to {output_name}")
    except Exception as e:
        QMessageBox.critical(window, "Error", f"An error occurred: {str(e)}")


def on_run_clicked():
    lat = float(lat_input.text())
    lon = float(lon_input.text())
    year = int(year_input.text())
    file_type = file_type_input.text()
    name = name_input.text()
    output_name = output_name_input.text()
    
    run_individual_location(output_name, lat, lon, year, file_type, name)


# Initialize the application
app = QApplication(sys.argv)

# Create the main window
window = QWidget()
window.setWindowTitle("NOAA MERRA2 Data Processor")
window.setGeometry(100, 100, 400, 300)

# Create a grid layout
layout = QGridLayout()

# Add widgets for input fields
layout.addWidget(QLabel("Latitude:"), 0, 0)
lat_input = QLineEdit()
layout.addWidget(lat_input, 0, 1)
lat_input.setText("41.766595")

layout.addWidget(QLabel("Longitude:"), 1, 0)
lon_input = QLineEdit()
layout.addWidget(lon_input, 1, 1)
lon_input.setText("-88.318735")

layout.addWidget(QLabel("Year:"), 2, 0)
year_input = QLineEdit()
layout.addWidget(year_input, 2, 1)
year_input.setText("2023")

layout.addWidget(QLabel("File Type:"), 3, 0)
file_type_input = QLineEdit()
layout.addWidget(file_type_input, 3, 1)
file_type_input.setText("AMY")

layout.addWidget(QLabel("Name:"), 4, 0)
name_input = QLineEdit()
layout.addWidget(name_input, 4, 1)
name_input.setText("TEST_2023")

layout.addWidget(QLabel("Output File Name:"), 5, 0)
output_name_input = QLineEdit()
layout.addWidget(output_name_input, 5, 1)
output_name_input.setText("TEST_2023.epw")

# Add a run button
run_button = QPushButton("Run")
run_button.clicked.connect(on_run_clicked)
layout.addWidget(run_button, 6, 0, 1, 2)

# Set the layout for the main window
window.setLayout(layout)

# Show the window
window.show()

# Run the application's main loop
sys.exit(app.exec_())


## Figure out Zip Codes

In [ ]:
import pandas as pd
import numpy as np

# Define a function to calculate the Haversine distance between two points in km
def haversine(lat1, lon1, lat2, lon2):
    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula to calculate the distance
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371  # Radius of Earth in kilometers
    return c * r

# 1) Open the file resources/zip_code_list.csv as a dataframe and call it zipcodes
zipcodes = pd.read_csv('resources/zip_codes_list.csv')

# 2) Open design_conditions.csv as a dataframe and call it dc
dc = pd.read_csv('resources/meteostat_stats.csv')

# Ensure latitude and longitude columns are in float format
zipcodes['lat'] = zipcodes['lat'].astype(float)
zipcodes['lng'] = zipcodes['lng'].astype(float)
dc['latitude'] = dc['latitude'].astype(float)
dc['longitude'] = dc['longitude'].astype(float)

# 3) Initialize columns in zipcodes dataframe for storing results
zipcodes['Distance'] = np.nan
zipcodes['Location'] = ""

# 4) Loop through all the rows in zipcodes
for idx, row in zipcodes.iterrows():
    lat1 = row['lat']
    lon1 = row['lng']

    # Calculate the distance to each location in the dc dataframe
    dc['Distance'] = dc.apply(lambda x: haversine(lat1, lon1, x['latitude'], x['longitude']), axis=1)

    # 5) Find the closest location in dc
    closest_location = dc.loc[dc['Distance'].idxmin()]

    # 6) Update the zipcodes dataframe with the closest location's details
    zipcodes.at[idx, 'Distance'] = closest_location['Distance']
    zipcodes.at[idx, 'Location'] = closest_location['name']

# Display the updated dataframe
zipcodes.to_csv('resources/updated_zip_code_list_again.csv', index=False)

In [ ]:
# Ensure the 'zip' column is a string and pad with zeros to make it 5 digits
zipcodes['zip0'] = zipcodes['zip'].astype(str).str.zfill(5)

zipcodes


In [ ]:
zipcodes.to_csv('resources/zip_code_list.csv', index=False)

In [ ]:
import pandas as pd
import requests
from io import StringIO

# URL of the dataset containing ZIP codes and their respective coordinates
url = "https://raw.githubusercontent.com/scpike/us-state-county-zip/master/geo-data.csv"

# Fetching the CSV file from the URL
response = requests.get(url)
response.raise_for_status()  # Raises an error for bad responses

# Reading the CSV data into a pandas DataFrame
data = pd.read_csv(StringIO(response.text))


data.to_csv('resources/zip_codes_list.csv')


In [ ]:
import pandas as pd
import requests
from io import BytesIO
from zipfile import ZipFile

# URL of a dataset containing ZIP codes, latitude, and longitude
url = "https://simplemaps.com/static/data/us-zips/1.74/basic/simplemaps_uszips_basicv1.74.zip"

# Download the ZIP file with SSL verification disabled
response = requests.get(url, verify=False)  # Bypass SSL certificate verification
response.raise_for_status()  # Check if the request was successful

# Unzip the file and read the CSV
with ZipFile(BytesIO(response.content)) as zip_file:
    # Extract the CSV file within the ZIP
    with zip_file.open('uszips.csv') as file:
        zip_code_data = pd.read_csv(file)

# # Display the DataFrame to verify the contents
# print(zip_code_data.head())

# # Save the DataFrame to a local CSV file
zip_code_data.to_csv('resources/zip_codes_list.csv', index=False)
# print("Data saved to 'us_zip_codes_with_coordinates.csv'.")


In [ ]:
zip_code_data